# 🚀 Foveated 2.5D Semantic Elevation Mapping — Colab GPU Deployment

**Smart India Hackathon (SIH26053) — Zero-Hardware Cloud GPU Demo**

This notebook deploys the full CUDA/TensorRT perception pipeline on a free Google Colab T4/L4 GPU, exposes the live dashboard via ngrok for evaluators, and supports smartphone camera integration via WebRTC.

| Metric | Value |
|--------|-------|
| GPU Architecture | NVIDIA T4 / L4 (16GB VRAM) |
| Cost | 100% Free |
| CUDA Kernel Latency | < 1.82 ms |
| VRAM Footprint | < 190 MB |
| Dashboard | Public ngrok URL |

> **⚠️ Before running:** Go to `Runtime → Change runtime type → T4 GPU`

## Step A: Verify GPU Runtime

In [ ]:
# Verify GPU is available and CUDA compiler is present
!nvidia-smi
!nvcc --version

import torch
print(f"\n✅ PyTorch CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    raise RuntimeError("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step B: Clone Repository & Install Dependencies

In [ ]:
import os

REPO_URL = "https://github.com/Kkushak16/Foveated-2.5D-Semantic-Elevation-Mapping.git"
REPO_DIR = "Foveated-2.5D-Semantic-Elevation-Mapping"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL}
else:
    print(f"ℹ️ Repository already cloned. Pulling latest...")
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

# Install core dependencies + Colab-specific packages
!pip install -q -r requirements.txt pynvml pyngrok psutil

## Step C: Build CUDA SIMT Projection Engine (CMake)

In [ ]:
%%bash
# Build the C++/CUDA grid projection engine natively on the Colab VM
# The Colab GPU (T4/L4) uses compute capability 7.5/8.9
mkdir -p build && cd build
cmake .. -DCMAKE_BUILD_TYPE=Release 2>&1 | tail -5
make -j$(nproc) 2>&1 | tail -10
echo ""
echo "✅ CUDA Grid Projection Engine built successfully!"

## Step D: Configure Ngrok Public Tunnel

This creates a public URL that SIH evaluators can access from any device.

> **Optional:** Set your ngrok auth token for longer sessions.  
> Get a free token at: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Optional: Set ngrok auth token for extended sessions (free account)
# Uncomment the line below and paste your token:
# NGROK_AUTH_TOKEN = "your_token_here"

NGROK_AUTH_TOKEN = None  # Set to your token string if you have one

from pyngrok import ngrok, conf

if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print("✅ Ngrok auth token set — extended session enabled")
else:
    print("ℹ️ No ngrok auth token set — using free tier (2-hour sessions)")
    print("   Get a free token: https://dashboard.ngrok.com/get-started/your-authtoken")

## Step E: Launch GPU Telemetry Collector

In [ ]:
# Validate the telemetry module works on this Colab GPU
import sys
sys.path.insert(0, 'python')
from telemetry import get_telemetry

telem = get_telemetry('cloud')
metrics = telem.get_metrics()

print("╔══════════════════════════════════════════════════════╗")
print("║        GPU TELEMETRY — COLAB T4/L4 VALIDATED        ║")
print("╠══════════════════════════════════════════════════════╣")
print(f"║  VRAM Used     : {metrics['vram_used_mb']:>8.1f} MB                    ║")
print(f"║  VRAM Total    : {metrics['vram_total_mb']:>8.1f} MB                    ║")
print(f"║  GPU Util      : {metrics['gpu_util_pct']:>8.1f} %                     ║")
print(f"║  Power Draw    : {metrics['power_draw_w']:>8.1f} W                     ║")
print(f"║  Temperature   : {metrics['soc_temp_c']:>8.1f} °C                    ║")
print(f"║  Platform      : {metrics['platform']:<25}       ║")
print("╚══════════════════════════════════════════════════════╝")

## Step F: Launch Dashboard Server with Ngrok Tunnel

This starts the full perception dashboard and creates a **public URL** for evaluators.

In [ ]:
import subprocess
import time
from pyngrok import ngrok

PORT = 8080

# Start the dashboard server in the background
# Uses Python http.server as Colab doesn't have Node.js by default
server_proc = subprocess.Popen(
    [sys.executable, '-m', 'http.server', str(PORT), '--directory', 'web/ui'],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

# Create ngrok tunnel
public_url = ngrok.connect(PORT)

print("")
print("╔══════════════════════════════════════════════════════════════════╗")
print("║               🌐 LIVE DEMO DASHBOARD IS ACTIVE                ║")
print("╠══════════════════════════════════════════════════════════════════╣")
print(f"║  Public URL    : {str(public_url):<46} ║")
print(f"║  Local URL     : http://localhost:{PORT:<39} ║")
print("║                                                                ║")
print("║  Share the Public URL with SIH evaluators!                     ║")
print("║  Open on your phone for live WebRTC camera streaming.          ║")
print("╚══════════════════════════════════════════════════════════════════╝")
print("")
print("📱 Smartphone Camera Setup:")
print(f"   1. Open {public_url} in Chrome/Safari on your phone")
print("   2. Click 'Launch Teleop HUD' on the landing page")
print("   3. Select '📹 Live Physical Webcam Feed' from Camera Sensor Input")
print("   4. Allow camera access when prompted")
print("   5. The pipeline processes frames in real-time with spatial masking + YOLO")

## Step G: YOLO Semantic Vision Server (Optional — Enhanced Detection)

Start the WebSocket-based YOLO server for real-time per-pixel object detection.  
The dashboard works without this (uses built-in browser cascade), but YOLO provides better accuracy.

In [ ]:
import subprocess

YOLO_PORT = 8090

yolo_proc = subprocess.Popen(
    [sys.executable, 'python/yolo_vision_server.py',
     '--port', str(YOLO_PORT), '--backend', 'auto'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT
)

time.sleep(5)  # Wait for model to load

print(f"✅ YOLO Vision Server started on ws://127.0.0.1:{YOLO_PORT}")
print("   Backend: auto (YOLOv8n → YOLOv5 → cv-cascade fallback)")

## Step H: Live Telemetry Monitor

Run this cell to see a live-updating telemetry readout while the demo is active.

In [ ]:
from IPython.display import clear_output
import time

telem = get_telemetry('cloud')
print("📊 Live Telemetry Monitor (Ctrl+M I to interrupt)")
print("=" * 55)

try:
    for i in range(120):  # Monitor for 10 minutes (5s intervals)
        metrics = telem.get_metrics()
        clear_output(wait=True)
        print(f"📊 Live GPU Telemetry — Sample {i+1}/120")
        print("=" * 55)
        print(f"  VRAM Usage    : {metrics['vram_used_mb']:.1f} / {metrics['vram_total_mb']:.1f} MB")
        print(f"  GPU Util      : {metrics['gpu_util_pct']:.1f}%")
        print(f"  Power Draw    : {metrics['power_draw_w']:.1f} W")
        print(f"  GPU Temp      : {metrics['soc_temp_c']:.1f} °C")
        print(f"  Platform      : {metrics['platform']}")
        print(f"  Dashboard     : {public_url}")
        
        # Warn if VRAM is high
        if metrics['vram_used_mb'] > 12000:
            print("\n⚠️  WARNING: VRAM usage above 12 GB — nearing T4 limit!")
        if metrics['soc_temp_c'] > 85:
            print("\n⚠️  WARNING: GPU temperature above 85°C — potential throttling!")
        
        time.sleep(5)
except KeyboardInterrupt:
    print("\n✅ Telemetry monitor stopped.")

## Step I: Quick Soak Test (5-minute validation)

Run a short soak test to verify the pipeline holds up under sustained load.

In [ ]:
!python scripts/soak_test.py --platform cloud --duration 300 --interval 5
print("\n✅ Soak test complete. Check the CSV log above for any thermal throttling events.")

## Cleanup: Stop Servers & Close Tunnel

In [ ]:
# Stop all servers and close the ngrok tunnel
ngrok.kill()
if 'server_proc' in dir() and server_proc:
    server_proc.terminate()
    print("✅ Dashboard server stopped")
if 'yolo_proc' in dir() and yolo_proc:
    yolo_proc.terminate()
    print("✅ YOLO vision server stopped")
print("✅ Ngrok tunnel closed")
print("\nAll services shut down cleanly.")